# Sampling Stability Experiment

random_state와 module별 stratified sampling 방식에 따라 Rule-Based(IQR)와 Isolation Forest 이상탐지 결과가 얼마나 달라지는지 검증합니다.

출력 파일은 `ml_outputs` 폴더에 저장됩니다.

In [4]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. 설정

- `FULL_CSV_PATH`: 전체 CSV 경로입니다. 기본값은 `C:\PdM_Project\rtu_data_full.csv`입니다.
- `SAMPLE_PER_MODULE`: seed마다 각 module에서 추출할 행 수입니다. module별 데이터가 이보다 적으면 가능한 전체 행을 사용합니다.
- `RANDOM_STATES`: 반복 실험 seed 목록입니다.

In [10]:
base_dir = Path.cwd()
output_dir = base_dir / "ml_outputs"
output_dir.mkdir(exist_ok=True)

FULL_CSV_PATH = Path(r"C:\Users\EL094\Downloads\rtu_data_full.csv")
# 전체 CSV 파일명이 다르거나 다른 위치에 있으면 위 경로만 수정하세요.

RANDOM_STATES = [0, 1, 7, 21, 42, 77, 100]
SAMPLE_PER_MODULE = 20_000

module_col = "module(equipment)"
time_col = "timestamp"

anomaly_feature_cols = [
    "activePower",
    "currentR", "currentS", "currentT",
    "voltageR", "voltageS", "voltageT",
]

iforest_feature_cols = [
    "activePower",
    "currentR", "currentS", "currentT",
    "voltageR", "voltageS", "voltageT",
    "powerFactorR", "powerFactorS", "powerFactorT",
]

print("input:", FULL_CSV_PATH)
print("output_dir:", output_dir)

input: C:\Users\EL094\Downloads\rtu_data_full.csv
output_dir: c:\PdM_Project\ml_outputs


## 2. 전체 CSV 로드 및 기본 전처리

In [11]:
def parse_timestamp_columns(frame):
    out = frame.copy()

    if "timestamp" in out.columns:
        ts_raw = out["timestamp"]
        if pd.api.types.is_numeric_dtype(ts_raw):
            out["timestamp"] = pd.to_datetime(ts_raw, unit="ms", errors="coerce")
        else:
            ts_as_num = pd.to_numeric(ts_raw, errors="coerce")
            parsed_as_epoch = pd.to_datetime(ts_as_num, unit="ms", errors="coerce")
            parsed_as_datetime = pd.to_datetime(ts_raw, errors="coerce")
            out["timestamp"] = parsed_as_epoch.where(ts_as_num.notna(), parsed_as_datetime)

    if "localtime" in out.columns:
        local_raw = out["localtime"]
        if pd.api.types.is_numeric_dtype(local_raw):
            local_text = local_raw.astype("Int64").astype(str)
        else:
            local_text = local_raw.astype(str).str.replace(r"\.0$", "", regex=True)

        parsed_by_format = pd.to_datetime(local_text, format="%Y%m%d%H%M%S", errors="coerce")
        parsed_general = pd.to_datetime(local_raw, errors="coerce")
        out["localtime"] = parsed_by_format.fillna(parsed_general)

    return out


def load_full_data(csv_path):
    if not csv_path.exists():
        raise FileNotFoundError(f"전체 CSV 파일을 찾을 수 없습니다: {csv_path}")

    df = pd.read_csv(csv_path)
    df = parse_timestamp_columns(df)

    required_cols = [module_col, time_col]
    missing_required = [c for c in required_cols if c not in df.columns]
    if missing_required:
        raise ValueError(f"필수 컬럼이 없습니다: {missing_required}")

    numeric_candidates = sorted(set(anomaly_feature_cols + iforest_feature_cols + ["accumActiveEnergy"]))
    for col in numeric_candidates:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[module_col, time_col]).copy()
    df[module_col] = df[module_col].astype(str)
    df = df.sort_values([module_col, time_col]).reset_index(drop=True)
    df["row_id"] = np.arange(len(df), dtype=np.int64)
    return df


df = load_full_data(FULL_CSV_PATH)
print("loaded shape:", df.shape)
display(df.head())
display(df.groupby(module_col).size().rename("rows").reset_index().sort_values("rows"))

loaded shape: (33696013, 20)


,module(equipment),timestamp,localtime,operation,voltageR,voltageS,voltageT,voltageRS,voltageST,voltageTR,currentR,currentS,currentT,activePower,powerFactorR,powerFactorS,powerFactorT,reactivePowerLagging,accumActiveEnergy,row_id
0,1(PM-3),2024-12-01 08:00:00,2024-12-01 00:00:00,1,214.38,214.45,219.10,371.37,375.45,375.39,15.16,15.53,20.65,2961.61,87.31,99.71,89.45,785.37,1955004,0
1,1(PM-3),2024-12-01 08:00:05,2024-12-01 00:00:05,1,214.05,211.74,218.68,368.73,372.74,374.74,25.70,7.07,19.75,3017.48,87.54,87.67,94.17,376.57,1955008,1
2,1(PM-3),2024-12-01 08:00:10,2024-12-01 00:00:10,1,215.79,214.92,211.10,373.00,368.94,369.69,13.64,14.87,13.70,2408.01,85.46,99.00,94.45,296.08,1955011,2
3,1(PM-3),2024-12-01 08:00:15,2024-12-01 00:00:15,1,210.39,214.92,215.57,368.32,372.80,368.89,25.76,26.35,5.80,3289.33,85.24,99.53,95.61,488.48,1955016,3
4,1(PM-3),2024-12-01 08:00:20,2024-12-01 00:00:20,1,216.71,216.37,215.65,375.05,374.13,374.43,8.65,29.49,15.09,3069.31,92.81,91.26,91.82,604.70,1955020,4


,module(equipment),rows
0,1(PM-3),2592001
1,11(우측분전반1),2592001
2,12(4호기),2592001
3,13(3호기),2592001
4,14(2호기),2592001
5,15(예비건조기),2592001
6,16(호이스트),2592001
7,17(6호기),2592001
8,18(우측분전반2),2592001
9,2(L-1전등),2592001


## 3. module별 stratified sampling

각 random_state마다 `groupby(module).sample()`을 사용합니다. 모든 module에서 동일한 최대 표본 수를 뽑되, 데이터가 부족한 module은 가능한 전체 행을 사용합니다.

In [12]:
def stratified_sample_by_module(frame, random_state, sample_per_module):
    work = frame.copy()
    if module_col not in work.columns and module_col in work.index.names:
        work = work.reset_index()

    if module_col not in work.columns:
        raise KeyError(f"'{module_col}' 컬럼을 찾을 수 없습니다. 현재 컬럼: {list(work.columns)}")

    sampled_parts = []
    for _, group in work.groupby(module_col, sort=False):
        n = min(len(group), sample_per_module)
        sampled_parts.append(group.sample(n=n, random_state=random_state, replace=False))

    sampled = pd.concat(sampled_parts, axis=0, ignore_index=True)
    sampled = sampled.sort_values([module_col, time_col]).reset_index(drop=True)
    return sampled


def sample_size_check(frame):
    return (
        frame.groupby(module_col)
        .size()
        .rename("sample_rows")
        .reset_index()
        .sort_values(module_col)
    )

preview_sample = stratified_sample_by_module(df, RANDOM_STATES[0], SAMPLE_PER_MODULE)
print("preview sample shape:", preview_sample.shape)
display(sample_size_check(preview_sample))

preview sample shape: (260000, 20)


,module(equipment),sample_rows
0,1(PM-3),20000
1,11(우측분전반1),20000
2,12(4호기),20000
3,13(3호기),20000
4,14(2호기),20000
5,15(예비건조기),20000
6,16(호이스트),20000
7,17(6호기),20000
8,18(우측분전반2),20000
9,2(L-1전등),20000


## 4. 이상탐지 함수 정의

In [13]:
def detect_iqr_rule(sample_df):
    features = [c for c in anomaly_feature_cols if c in sample_df.columns]
    if not features:
        raise ValueError("Rule-Based 이상탐지에 사용할 feature가 없습니다.")

    rule_df = sample_df[["row_id", module_col, time_col] + features].copy()
    rule_df["rule_anomaly"] = 0
    rule_df["rule_reason"] = "normal"

    for col in features:
        q1 = rule_df.groupby(module_col)[col].transform(lambda s: s.quantile(0.25))
        q3 = rule_df.groupby(module_col)[col].transform(lambda s: s.quantile(0.75))
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        low_flag = rule_df[col] < lower
        high_flag = rule_df[col] > upper
        flag = low_flag | high_flag

        rule_df.loc[flag, "rule_anomaly"] = 1
        rule_df.loc[low_flag, "rule_reason"] = rule_df.loc[low_flag, "rule_reason"] + f"|{col}_low"
        rule_df.loc[high_flag, "rule_reason"] = rule_df.loc[high_flag, "rule_reason"] + f"|{col}_high"

    rule_df["rule_reason"] = rule_df["rule_reason"].str.strip("|")
    return rule_df


def detect_isolation_forest(sample_df, random_state, contamination=0.01):
    features = [c for c in iforest_feature_cols if c in sample_df.columns]
    if not features:
        raise ValueError("Isolation Forest 이상탐지에 사용할 feature가 없습니다.")

    base_cols = ["row_id", module_col, time_col] + features
    ml_df = sample_df[base_cols].dropna(subset=features).copy()
    ml_df["hour"] = ml_df[time_col].dt.hour
    ml_df["dayofweek"] = ml_df[time_col].dt.dayofweek

    model_df = pd.get_dummies(ml_df[[module_col] + features + ["hour", "dayofweek"]], columns=[module_col], drop_first=False)
    feature_matrix = model_df.astype(float)

    scaler = StandardScaler()
    x_scaled = scaler.fit_transform(feature_matrix)

    model = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=random_state,
        n_jobs=-1,
    )

    ml_df["iforest_pred"] = model.fit_predict(x_scaled)
    ml_df["iforest_anomaly"] = (ml_df["iforest_pred"] == -1).astype(int)
    ml_df["iforest_score"] = model.decision_function(x_scaled)
    return ml_df

## 5. seed별 반복 실험 실행

In [14]:
def count_by_module(result_df, flag_col, count_col, random_state):
    counts = (
        result_df[result_df[flag_col] == 1]
        .groupby(module_col)
        .size()
        .rename(count_col)
        .reset_index()
    )

    modules = pd.DataFrame({module_col: sorted(df[module_col].unique())})
    counts = modules.merge(counts, on=module_col, how="left").fillna({count_col: 0})
    counts[count_col] = counts[count_col].astype(int)
    counts["random_state"] = random_state
    return counts[["random_state", module_col, count_col]]


def add_rank(counts_df, count_col, rank_col):
    out = counts_df.copy()
    out[rank_col] = (
        out.groupby("random_state")[count_col]
        .rank(method="min", ascending=False)
        .astype(int)
    )
    return out


def calculate_overlap(rule_df, iforest_df, random_state):
    rule_flags = rule_df[["row_id", module_col, "rule_anomaly"]].copy()
    iforest_flags = iforest_df[["row_id", "iforest_anomaly"]].copy()

    merged = rule_flags.merge(iforest_flags, on="row_id", how="left")
    merged["iforest_anomaly"] = merged["iforest_anomaly"].fillna(0).astype(int)

    merged["both_flag"] = ((merged["rule_anomaly"] == 1) & (merged["iforest_anomaly"] == 1)).astype(int)
    merged["rule_only_flag"] = ((merged["rule_anomaly"] == 1) & (merged["iforest_anomaly"] == 0)).astype(int)
    merged["iforest_only_flag"] = ((merged["rule_anomaly"] == 0) & (merged["iforest_anomaly"] == 1)).astype(int)

    overlap = (
        merged.groupby(module_col)
        .agg(
            rule_count=("rule_anomaly", "sum"),
            iforest_count=("iforest_anomaly", "sum"),
            both_count=("both_flag", "sum"),
            rule_only_count=("rule_only_flag", "sum"),
            iforest_only_count=("iforest_only_flag", "sum"),
        )
        .reset_index()
    )
    overlap["overlap_count"] = overlap["both_count"]
    union_count = overlap["rule_count"] + overlap["iforest_count"] - overlap["both_count"]
    overlap["overlap_ratio_of_rule"] = np.where(overlap["rule_count"] > 0, overlap["both_count"] / overlap["rule_count"], 0)
    overlap["overlap_ratio_of_iforest"] = np.where(overlap["iforest_count"] > 0, overlap["both_count"] / overlap["iforest_count"], 0)
    overlap["jaccard_overlap_ratio"] = np.where(union_count > 0, overlap["both_count"] / union_count, 0)
    overlap["random_state"] = random_state

    cols = [
        "random_state", module_col,
        "rule_count", "iforest_count",
        "overlap_count", "rule_only_count", "iforest_only_count", "both_count",
        "overlap_ratio_of_rule", "overlap_ratio_of_iforest", "jaccard_overlap_ratio",
    ]
    return overlap[cols]


rule_count_parts = []
iforest_count_parts = []
overlap_parts = []

for seed in RANDOM_STATES:
    sampled = stratified_sample_by_module(df, seed, SAMPLE_PER_MODULE)

    rule_result = detect_iqr_rule(sampled)
    iforest_result = detect_isolation_forest(sampled, random_state=seed, contamination=0.01)

    rule_counts = count_by_module(rule_result, "rule_anomaly", "rule_count", seed)
    iforest_counts = count_by_module(iforest_result, "iforest_anomaly", "iforest_count", seed)
    overlap = calculate_overlap(rule_result, iforest_result, seed)

    rule_count_parts.append(rule_counts)
    iforest_count_parts.append(iforest_counts)
    overlap_parts.append(overlap)

    print(
        f"seed={seed} | sample_rows={len(sampled):,} | "
        f"rule={int(rule_counts['rule_count'].sum()):,} | "
        f"iforest={int(iforest_counts['iforest_count'].sum()):,} | "
        f"both={int(overlap['both_count'].sum()):,}"
    )

rule_counts_all = add_rank(pd.concat(rule_count_parts, ignore_index=True), "rule_count", "rule_rank")
iforest_counts_all = add_rank(pd.concat(iforest_count_parts, ignore_index=True), "iforest_count", "iforest_rank")
overlap_all = pd.concat(overlap_parts, ignore_index=True)

combined_counts = (
    rule_counts_all
    .merge(iforest_counts_all, on=["random_state", module_col], how="outer")
    .merge(overlap_all, on=["random_state", module_col, "rule_count", "iforest_count"], how="left")
)

display(combined_counts.sort_values(["random_state", "rule_rank", "iforest_rank"]).head(30))

seed=0 | sample_rows=260,000 | rule=142 | iforest=2,600 | both=51
seed=1 | sample_rows=260,000 | rule=175 | iforest=2,600 | both=61
seed=7 | sample_rows=260,000 | rule=166 | iforest=2,600 | both=68
seed=21 | sample_rows=260,000 | rule=156 | iforest=2,600 | both=60
seed=42 | sample_rows=260,000 | rule=149 | iforest=2,600 | both=60
seed=77 | sample_rows=260,000 | rule=170 | iforest=2,600 | both=37
seed=100 | sample_rows=260,000 | rule=151 | iforest=2,600 | both=84


,random_state,module(equipment),rule_count,rule_rank,iforest_count,iforest_rank,overlap_count,rule_only_count,iforest_only_count,both_count,overlap_ratio_of_rule,overlap_ratio_of_iforest,jaccard_overlap_ratio
3,0,13(3호기),85,1,76,11,13,72,63,13,0.152941,0.171053,0.087838
1,0,11(우측분전반1),9,2,84,10,6,3,78,6,0.666667,0.071429,0.068966
2,0,12(4호기),7,3,344,3,6,1,338,6,0.857143,0.017442,0.017391
7,0,17(6호기),6,4,559,2,5,1,554,5,0.833333,0.008945,0.008929
0,0,1(PM-3),6,4,119,8,2,4,117,2,0.333333,0.016807,0.016260
5,0,15(예비건조기),5,6,183,4,2,3,181,2,0.400000,0.010929,0.010753
8,0,18(우측분전반2),5,6,129,7,2,3,127,2,0.400000,0.015504,0.015152
12,0,5(좌측분전반),5,6,73,12,5,0,68,5,1.000000,0.068493,0.068493
4,0,14(2호기),4,9,568,1,3,1,565,3,0.750000,0.005282,0.005272
9,0,2(L-1전등),4,9,160,5,3,1,157,3,0.750000,0.018750,0.018634


## 6. module별 평균, 표준편차, 순위 변동 계산

In [15]:
summary = (
    combined_counts.groupby(module_col)
    .agg(
        rule_count_mean=("rule_count", "mean"),
        rule_count_std=("rule_count", "std"),
        rule_count_min=("rule_count", "min"),
        rule_count_max=("rule_count", "max"),
        rule_rank_mean=("rule_rank", "mean"),
        rule_rank_std=("rule_rank", "std"),
        rule_rank_min=("rule_rank", "min"),
        rule_rank_max=("rule_rank", "max"),
        iforest_count_mean=("iforest_count", "mean"),
        iforest_count_std=("iforest_count", "std"),
        iforest_count_min=("iforest_count", "min"),
        iforest_count_max=("iforest_count", "max"),
        iforest_rank_mean=("iforest_rank", "mean"),
        iforest_rank_std=("iforest_rank", "std"),
        iforest_rank_min=("iforest_rank", "min"),
        iforest_rank_max=("iforest_rank", "max"),
        both_count_mean=("both_count", "mean"),
        both_count_std=("both_count", "std"),
        overlap_ratio_of_rule_mean=("overlap_ratio_of_rule", "mean"),
        overlap_ratio_of_iforest_mean=("overlap_ratio_of_iforest", "mean"),
        jaccard_overlap_ratio_mean=("jaccard_overlap_ratio", "mean"),
    )
    .reset_index()
)

summary["rule_count_range"] = summary["rule_count_max"] - summary["rule_count_min"]
summary["iforest_count_range"] = summary["iforest_count_max"] - summary["iforest_count_min"]
summary["rule_rank_range"] = summary["rule_rank_max"] - summary["rule_rank_min"]
summary["iforest_rank_range"] = summary["iforest_rank_max"] - summary["iforest_rank_min"]
summary["rule_count_cv"] = np.where(summary["rule_count_mean"] > 0, summary["rule_count_std"] / summary["rule_count_mean"], 0)
summary["iforest_count_cv"] = np.where(summary["iforest_count_mean"] > 0, summary["iforest_count_std"] / summary["iforest_count_mean"], 0)

summary = summary.sort_values(["iforest_count_mean", "rule_count_mean"], ascending=False).reset_index(drop=True)
display(summary)

,module(equipment),rule_count_mean,rule_count_std,rule_count_min,rule_count_max,rule_rank_mean,rule_rank_std,rule_rank_min,rule_rank_max,iforest_count_mean,iforest_count_std,iforest_count_min,iforest_count_max,iforest_rank_mean,iforest_rank_std,iforest_rank_min,iforest_rank_max,both_count_mean,both_count_std,overlap_ratio_of_rule_mean,overlap_ratio_of_iforest_mean,jaccard_overlap_ratio_mean,rule_count_range,iforest_count_range,rule_rank_range,iforest_rank_range,rule_count_cv,iforest_count_cv
0,17(6호기),5.285714,3.147183,1,10,6.857143,3.891382,2,11,340.714286,173.512453,89,559,4.000000,3.741657,1,11,3.571429,2.225395,0.713095,0.011508,0.011387,9,470,9,10,0.595413,0.509261
1,15(예비건조기),5.571429,2.992053,3,12,6.285714,2.360387,2,9,276.714286,99.565245,164,430,3.857143,1.951800,2,7,2.857143,1.772811,0.504762,0.011186,0.011003,9,266,7,5,0.537035,0.359812
2,12(4호기),4.142857,3.184785,0,9,7.571429,3.690399,3,12,236.285714,111.160116,120,359,5.857143,4.140393,1,12,3.428571,2.572751,0.728798,0.015169,0.015121,9,239,9,11,0.768741,0.470448
3,14(2호기),5.428571,2.299068,3,10,6.000000,2.645751,3,9,233.000000,166.575308,73,568,6.142857,4.375255,1,12,2.857143,0.899735,0.578571,0.016736,0.016467,7,495,6,11,0.423513,0.714915
4,5(좌측분전반),4.000000,1.632993,2,7,8.142857,2.267787,5,12,226.285714,114.354295,73,426,5.857143,3.532165,1,12,3.000000,1.000000,0.799320,0.020642,0.020590,5,353,7,11,0.408248,0.505354
5,18(우측분전반2),3.571429,1.812654,0,5,7.857143,2.544836,5,12,220.428571,119.590770,129,464,5.714286,2.690371,2,9,1.571429,1.397276,0.380952,0.007424,0.007309,5,335,7,7,0.507543,0.542538
6,2(L-1전등),7.285714,3.093773,4,11,4.857143,2.968084,2,9,186.000000,184.925210,77,599,8.428571,4.157609,1,13,3.285714,1.889822,0.463245,0.023354,0.022164,7,522,7,12,0.424635,0.994222
7,1(PM-3),5.428571,2.149197,3,8,5.714286,2.811541,3,9,181.714286,112.029333,66,359,7.285714,4.151879,1,13,2.571429,1.511858,0.458333,0.017870,0.017465,5,293,6,12,0.395905,0.616514
8,13(3호기),99.285714,11.743286,85,115,1.000000,0.000000,1,1,165.428571,96.410333,25,290,7.428571,3.823486,3,13,28.571429,15.197118,0.294790,0.192571,0.115695,30,265,0,10,0.118278,0.582791
9,11(우측분전반1),5.285714,3.251373,1,9,6.428571,4.466809,2,13,158.714286,89.853055,64,293,8.000000,3.559026,3,13,2.714286,1.799471,0.577381,0.027538,0.026424,8,229,11,10,0.615125,0.566131


## 7. 13(3호기) seed 민감도 확인

특정 random_state에서만 `13(3호기)`가 높게 나오는지 직접 확인하기 위한 셀입니다.

In [16]:
target_pattern = "13(3호기)"

target_rows = combined_counts[combined_counts[module_col].str.contains(target_pattern, regex=False, na=False)].copy()
if target_rows.empty:
    print(f"대상 module을 찾지 못했습니다: {target_pattern}")
else:
    display(target_rows.sort_values("random_state"))

    target_summary = summary[summary[module_col].str.contains(target_pattern, regex=False, na=False)].copy()
    display(target_summary)

,random_state,module(equipment),rule_count,rule_rank,iforest_count,iforest_rank,overlap_count,rule_only_count,iforest_only_count,both_count,overlap_ratio_of_rule,overlap_ratio_of_iforest,jaccard_overlap_ratio
3,0,13(3호기),85,1,76,11,13,72,63,13,0.152941,0.171053,0.087838
16,1,13(3호기),110,1,128,10,27,83,101,27,0.245455,0.210938,0.127962
29,7,13(3호기),109,1,161,5,32,77,129,32,0.293578,0.198758,0.134454
42,21,13(3호기),90,1,290,3,32,58,258,32,0.355556,0.110345,0.091954
55,42,13(3호기),92,1,251,5,36,56,215,36,0.391304,0.143426,0.117264
68,77,13(3호기),115,1,25,13,7,108,18,7,0.060870,0.280000,0.052632
81,100,13(3호기),94,1,227,5,53,41,174,53,0.563830,0.233480,0.197761


,module(equipment),rule_count_mean,rule_count_std,rule_count_min,rule_count_max,rule_rank_mean,rule_rank_std,rule_rank_min,rule_rank_max,iforest_count_mean,iforest_count_std,iforest_count_min,iforest_count_max,iforest_rank_mean,iforest_rank_std,iforest_rank_min,iforest_rank_max,both_count_mean,both_count_std,overlap_ratio_of_rule_mean,overlap_ratio_of_iforest_mean,jaccard_overlap_ratio_mean,rule_count_range,iforest_count_range,rule_rank_range,iforest_rank_range,rule_count_cv,iforest_count_cv
8,13(3호기),99.285714,11.743286,85,115,1.0,0.0,1,1,165.428571,96.410333,25,290,7.428571,3.823486,3,13,28.571429,15.197118,0.29479,0.192571,0.115695,30,265,0,10,0.118278,0.582791


## 8. 결과 CSV 저장

In [17]:
rule_counts_path = output_dir / "sampling_stability_rule_counts.csv"
iforest_counts_path = output_dir / "sampling_stability_iforest_counts.csv"
summary_path = output_dir / "sampling_stability_summary.csv"
overlap_path = output_dir / "rule_vs_iforest_overlap_by_seed.csv"

rule_counts_all.rename(columns={module_col: "module"}).to_csv(rule_counts_path, index=False, encoding="utf-8-sig")
iforest_counts_all.rename(columns={module_col: "module"}).to_csv(iforest_counts_path, index=False, encoding="utf-8-sig")
summary.rename(columns={module_col: "module"}).to_csv(summary_path, index=False, encoding="utf-8-sig")
combined_counts.rename(columns={module_col: "module"}).to_csv(overlap_path, index=False, encoding="utf-8-sig")

print("saved:")
print(rule_counts_path)
print(iforest_counts_path)
print(summary_path)
print(overlap_path)

saved:
c:\PdM_Project\ml_outputs\sampling_stability_rule_counts.csv
c:\PdM_Project\ml_outputs\sampling_stability_iforest_counts.csv
c:\PdM_Project\ml_outputs\sampling_stability_summary.csv
c:\PdM_Project\ml_outputs\rule_vs_iforest_overlap_by_seed.csv
